# V6 True Event-by-Event Online State Replay

This CPU notebook reconstructs short-term user state one event at a time and compares it with the locked A2 Gold snapshots. For every timestamp `t`, all queries read state from events `< t`; only afterward are events at `t` appended.

Use a **High-RAM CPU** runtime. GPU is not used. The large temporary SQLite spool stays on Colab local disk; Drive receives only the small report bundle.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_v6_true_online_state')
SOURCE_ROOT = WORK_ROOT / 'runtime'
REPORT_DIR = WORK_ROOT / 'report'
WORK_DB_DIR = Path('/content/recsys_online_state_work')
DRIVE_REPORT_DIR = DRIVE_ROOT / 'artifacts/adaptive/v6_true_online_state'
for path in (SOURCE_ROOT, REPORT_DIR, DRIVE_REPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)
print('adaptive root:', ADAPTIVE_ROOT)
print('protocol:', PROTOCOL_DIR)
print('temporary work:', WORK_DB_DIR)

In [ ]:
required = {'pyarrow': 'pyarrow', 'pandas': 'pandas', 'huggingface_hub': 'huggingface_hub>=0.32'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    PROTOCOL_DIR / 'queries.parquet',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing inputs:\n' + '\n'.join(missing_paths)
free_gib = shutil.disk_usage('/content').free / 1024**3
assert free_gib >= 8, f'At least 8 GiB local disk is required; only {free_gib:.1f} GiB is free.'
print(f'input preflight: PASS | local disk free: {free_gib:.1f} GiB')

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_FILENAME = 'runtime/adaptive_online_state_runtime_v1.zip'
RUNTIME_EXPECTED_SHA256 = 'b35377c0e113ba44b2f0acba1852337132458765799ca13fca514a77163663c8'

# Normalize Colab Secret values because some Colab versions return a wrapped dict.
def normalize_hf_token(raw):
    if isinstance(raw, bytes):
        raw = raw.decode('utf-8')
    if isinstance(raw, str):
        raw = raw.strip()
        return raw if raw.startswith('hf_') else None
    if isinstance(raw, dict):
        for value in raw.values():
            token = normalize_hf_token(value)
            if token:
                return token
    if isinstance(raw, (list, tuple)):
        for value in raw:
            token = normalize_hf_token(value)
            if token:
                return token
    return None

try:
    HF_TOKEN = normalize_hf_token(userdata.get('HF_TOKEN'))
except Exception as exc:
    print('Colab Secret unavailable:', type(exc).__name__)
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = normalize_hf_token(getpass('Paste Hugging Face READ token: '))
assert HF_TOKEN, 'HF_TOKEN must begin with hf_.'

runtime_zip = Path(hf_hub_download(
    repo_id=HF_REPO_ID, filename=RUNTIME_FILENAME, repo_type='model', token=HF_TOKEN
))

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
REPLAY_SCRIPT = SOURCE_ROOT / 'scripts/run_adaptive_v6_true_online_state.py'
assert REPLAY_SCRIPT.exists()
compile(REPLAY_SCRIPT.read_text(), str(REPLAY_SCRIPT), 'exec')
print('runtime integrity: PASS')

## Run the state reducer

The shard scan can be quiet between progress messages. It reads train and validation once, keeps only holdout users, and writes a local disk-backed event spool so RAM does not grow with the full history.

In [ ]:
if REPORT_DIR.exists():
    shutil.rmtree(REPORT_DIR)
REPORT_DIR.mkdir(parents=True)
if WORK_DB_DIR.exists():
    shutil.rmtree(WORK_DB_DIR)

command = [
    sys.executable, str(REPLAY_SCRIPT),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--protocol-dir', str(PROTOCOL_DIR),
    '--output-dir', str(REPORT_DIR),
    '--work-dir', str(WORK_DB_DIR),
    '--max-sequence-len', '30',
    '--max-session-sequence-len', '10',
    '--calibration-user-fraction', '0.35',
    '--seed', '44',
]
log_path = Path('/content/v6_true_online_state.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w') as log:
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-200:])
    raise RuntimeError(f'Online-state replay failed with exit code {return_code}:\n{tail}')
required_outputs = [
    REPORT_DIR / 'online_state_parity_report.json',
    REPORT_DIR / 'online_state_parity_per_query.csv',
    REPORT_DIR / 'online_state_mismatch_examples.json',
]
assert all(path.exists() for path in required_outputs), 'Replay finished without all reports.'
print('replay artifacts: PASS')

In [ ]:
report = json.loads((REPORT_DIR / 'online_state_parity_report.json').read_text())
print(json.dumps(report, indent=2))
print('\nACCEPTED:', report['parity']['accepted'])
print('core parity:', f"{report['parity']['core_fields_match_rate']:.4%}")
print('state events/sec:', f"{report['timing']['state_events_per_second']:,.0f}")

In [ ]:
bundle_base = Path('/content/v6_true_online_state_report')
bundle_path = Path(shutil.make_archive(str(bundle_base), 'zip', REPORT_DIR))
drive_bundle = DRIVE_REPORT_DIR / bundle_path.name
shutil.copy2(bundle_path, drive_bundle)
print('Drive backup:', drive_bundle, f'{drive_bundle.stat().st_size / 1024**2:.2f} MiB')
files.download(str(bundle_path))

## Interpretation

- `accepted=true` means the event-by-event reducer exactly reproduces all core recent and session arrays/counts for the A2 holdout queries.
- Scalar rate/time fields are also reported, but they are diagnostic because the original Gold builder computed some of them before its same-timestamp sanitation step.
- This step validates online state mutation. It intentionally does not repeat the expensive exact-matrix V4.2 retrieval: once input parity passes, the model receives the same adaptive sequence tensors already evaluated in notebook 26.
- Long-term aggregate reducers remain a separate production task; this notebook reconstructs the bounded short-term and session state consumed by V4.